# Episode 13 — Your own Verilog block over AXI-Lite

To dim an LED you switch it on and off very fast (**PWM**, pulse-width modulation) and change
the fraction of time it is on. At 1 kHz that means a precise edge every few microseconds,
which Python on Linux cannot do steadily. Hardware can, so we write a PWM block in Verilog
and give it **registers** that Python writes over AXI-Lite.

```
 Python --AXI-Lite--> edge_pwm registers --> 4 PWM counters --> 4 LEDs
```

## The block: `hdl/edge_pwm.v`

| Offset | Register | Meaning |
| --- | --- | --- |
| 0x00 | ID | always `0x50574D31` ("PWM1"), to check you found the right block |
| 0x04 | ENABLE | bit *n* = 1 turns channel *n* on |
| 0x08 | PERIOD | PWM period in clock cycles (100 MHz clock: 100 000 = 1 kHz) |
| 0x10–0x1C | DUTY0–3 | on-time of each channel in clock cycles |
| 0x20 | COUNT | the running period counter (read only) |

Inside, an **AXI-Lite slave** answers the processor's reads and writes (address, data and
response handshakes), and a **counter** runs from 0 to PERIOD−1. Each output is high while
the counter is below that channel's DUTY. New PERIOD and DUTY values are copied in at the
start of the next period, so a change never cuts a pulse short.

The testbench `sim/tb_edge_pwm.sv` checks all of this before it goes near the board
(`sim/run_xsim.bat` runs it in the Vivado simulator and prints `TEST PASSED`).

## Building the overlay

`vivado/pwm_overlay.tcl` adds the Verilog file to a block design as a **module reference**,
connects it to the processor, and adds an AXI GPIO for the switches and buttons:

```
vivado -mode batch -source vivado/pwm_overlay.tcl -tclargs xc7z020clg400-1 work overlays/z20
```

In [ ]:
from pynq import Overlay
from pynq.overlays.base import board_variant
import time

chip = {"EDGE-Z10": "z10", "EDGE-Z20": "z20"}[board_variant()]
ol = Overlay("overlays/{}/pwm_overlay.bit".format(chip))

for name, ip in ol.ip_dict.items():
    if "phys_addr" not in ip:          # the processor itself has no address
        continue
    print("{:12s} 0x{:08X}  {}".format(name, ip["phys_addr"], ip["type"]))

## 1. Find the block and read its ID

PYNQ has no driver for our block, so it gives us a plain `DefaultIP` with `read()` and
`write()` at byte offsets.

In [ ]:
pwm = ol.pwm_0
print(type(pwm).__name__)

ident = pwm.read(0x00)
print(hex(ident), ident.to_bytes(4, "big").decode())

## 2. Write the registers

In [ ]:
ENABLE, PERIOD, DUTY0, COUNT = 0x04, 0x08, 0x10, 0x20
CLOCK_HZ = 100_000_000

period = CLOCK_HZ // 1000                     # 1 kHz
pwm.write(PERIOD, period)
for ch, percent in enumerate([5, 25, 50, 100]):
    pwm.write(DUTY0 + 4 * ch, period * percent // 100)
pwm.write(ENABLE, 0b1111)

print("PERIOD", pwm.read(PERIOD), "| DUTY", [pwm.read(DUTY0 + 4 * ch) for ch in range(4)])
print("counter now at", pwm.read(COUNT))

The four LEDs now glow at four different brightness levels, and Python is doing nothing.

## 3. A breathing LED

Our eyes see brightness roughly on a log scale, so a **gamma** curve (`level²`) makes the
fade look smooth. Runs for 10 seconds.

In [ ]:
import math

def set_percent(ch, percent):
    pwm.write(DUTY0 + 4 * ch, int(period * percent / 100))

t0 = time.time()
while time.time() - t0 < 10:
    level = (1 - math.cos(2 * math.pi * (time.time() - t0) / 2.0)) / 2     # 0..1, 2 s cycle
    for ch in range(4):
        set_percent(ch, 100 * level ** 2)
    time.sleep(0.02)

## 4. Slide switches choose the brightness

The AXI GPIO's channel 1 holds the switches. The switch value 0–15 sets all four LEDs, for 15 seconds.

In [ ]:
switches = ol.inputs_gpio.channel1
switches.setlength(4)
switches.setdirection("in")

end = time.time() + 15
while time.time() < end:
    value = switches.read()
    for ch in range(4):
        set_percent(ch, 100 * (value / 15) ** 2)
    time.sleep(0.05)

## 5. Slow it down to see the PWM

With a period of 1 second (100 million clock cycles) and 50 % duty, the LEDs simply blink.

In [ ]:
pwm.write(PERIOD, CLOCK_HZ)
for ch in range(4):
    pwm.write(DUTY0 + 4 * ch, CLOCK_HZ * (ch + 1) // 5)     # 20, 40, 60, 80 %
time.sleep(5)
pwm.write(PERIOD, period)                                     # back to 1 kHz
for ch in range(4):
    set_percent(ch, 25)

## Try it yourself

Use the two push buttons (`ol.inputs_gpio.channel2`) to make the brightness go up and down
in steps of 10 %.

In [ ]:
# your code here
pwm.write(ENABLE, 0)